In [1]:
# ============================================================
# HOUSE PRICE PREDICTION
# COMPLETE DATA PREPROCESSING
# ============================================================

# ------------------------------------------------------------
# STEP 1: IMPORT LIBRARIES
# ------------------------------------------------------------

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ------------------------------------------------------------
# STEP 2: LOAD DATASET
# ------------------------------------------------------------

# If using Google Colab:
# Upload your CSV first using:
#
# from google.colab import files
# uploaded = files.upload()
#
# Then use the filename below.

file_path = "train.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Original shape:", df.shape)


# ------------------------------------------------------------
# STEP 3: VIEW DATA
# ------------------------------------------------------------

print("\nFirst 5 rows:")
print(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
print(df.info())

print("\nStatistical summary:")
print(df.describe(include="all"))


# ------------------------------------------------------------
# STEP 4: CHECK MISSING VALUES
# ------------------------------------------------------------

print("\nMissing values:")
print(df.isnull().sum())


# ------------------------------------------------------------
# STEP 5: CHECK DUPLICATES
# ------------------------------------------------------------

print("\nNumber of duplicate rows:")
print(df.duplicated().sum())


# ------------------------------------------------------------
# STEP 6: REMOVE DUPLICATES
# ------------------------------------------------------------

df = df.drop_duplicates()

df = df.reset_index(drop=True)

print("\nShape after removing duplicates:")
print(df.shape)


# ------------------------------------------------------------
# STEP 7: CLEAN COLUMN NAMES
# ------------------------------------------------------------

df.columns = (
    df.columns
    .str.strip()
)


# ------------------------------------------------------------
# STEP 8: CONVERT NUMERIC COLUMNS
# ------------------------------------------------------------

numeric_columns = [
    "UNDER_CONSTRUCTION",
    "RERA",
    "BHK_NO.",
    "SQUARE_FT",
    "READY_TO_MOVE",
    "RESALE",
    "LONGITUDE",
    "LATITUDE",
    "TARGET(PRICE_IN_LACS)"
]

for column in numeric_columns:

    if column in df.columns:

        df[column] = pd.to_numeric(
            df[column],
            errors="coerce"
        )


# ------------------------------------------------------------
# STEP 9: HANDLE INVALID VALUES
# ------------------------------------------------------------

# Square feet cannot be zero or negative

df.loc[
    df["SQUARE_FT"] <= 0,
    "SQUARE_FT"
] = np.nan


# BHK cannot be zero or negative

df.loc[
    df["BHK_NO."] <= 0,
    "BHK_NO."
] = np.nan


# Price cannot be zero or negative

df.loc[
    df["TARGET(PRICE_IN_LACS)"] <= 0,
    "TARGET(PRICE_IN_LACS)"
] = np.nan


# ------------------------------------------------------------
# STEP 10: FIX LATITUDE AND LONGITUDE
# ------------------------------------------------------------

# The uploaded dataset has latitude/longitude values
# appearing in the opposite columns.
#
# Swap them.

df[
    ["LATITUDE", "LONGITUDE"]
] = df[
    ["LONGITUDE", "LATITUDE"]
]


# ------------------------------------------------------------
# STEP 11: VALIDATE GEOGRAPHIC VALUES
# ------------------------------------------------------------

valid_coordinates = (
    df["LATITUDE"].between(6, 38) &
    df["LONGITUDE"].between(68, 98)
)


# Invalid coordinates are converted to missing values.

df.loc[
    ~valid_coordinates,
    ["LATITUDE", "LONGITUDE"]
] = np.nan


print(
    "\nInvalid geographic coordinates:",
    (~valid_coordinates).sum()
)


# ------------------------------------------------------------
# STEP 12: HANDLE ADDRESS COLUMN
# ------------------------------------------------------------

# Fill missing address values

df["ADDRESS"] = df["ADDRESS"].fillna(
    "Unknown"
)


# Split address into locality and city

address_parts = df[
    "ADDRESS"
].str.split(
    ",",
    n=1,
    expand=True
)


df["LOCALITY"] = (
    address_parts[0]
    .str.strip()
)


if address_parts.shape[1] > 1:

    df["CITY"] = (
        address_parts[1]
        .str.strip()
    )

else:

    df["CITY"] = "Unknown"


# Replace empty values

df["LOCALITY"] = (
    df["LOCALITY"]
    .replace("", "Unknown")
)

df["CITY"] = (
    df["CITY"]
    .replace("", "Unknown")
)


# ------------------------------------------------------------
# STEP 13: ENCODE LOCALITY
# ------------------------------------------------------------

# There are many different localities.
#
# Instead of creating thousands of dummy variables,
# use frequency encoding.

locality_frequency = (
    df["LOCALITY"]
    .value_counts(normalize=True)
)


df["LOCALITY_FREQUENCY"] = (
    df["LOCALITY"]
    .map(locality_frequency)
    .fillna(0)
)


# Remove original ADDRESS and LOCALITY

df = df.drop(
    columns=[
        "ADDRESS",
        "LOCALITY"
    ]
)


# ------------------------------------------------------------
# STEP 14: FEATURE ENGINEERING
# ------------------------------------------------------------

# Area per BHK

df["AREA_PER_BHK"] = (
    df["SQUARE_FT"] /
    df["BHK_NO."]
)


# Replace infinite values

df["AREA_PER_BHK"] = (
    df["AREA_PER_BHK"]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)


# ------------------------------------------------------------
# STEP 15: DEFINE TARGET
# ------------------------------------------------------------

target = "TARGET(PRICE_IN_LACS)"


# ------------------------------------------------------------
# STEP 16: REMOVE ROWS WITH MISSING TARGET
# ------------------------------------------------------------

df = df.dropna(
    subset=[target]
)


# ------------------------------------------------------------
# STEP 17: SEPARATE FEATURES AND TARGET
# ------------------------------------------------------------

X = df.drop(
    columns=[target]
)

y = df[target]


# ------------------------------------------------------------
# STEP 18: DEFINE NUMERIC FEATURES
# ------------------------------------------------------------

numeric_features = [
    "UNDER_CONSTRUCTION",
    "RERA",
    "BHK_NO.",
    "SQUARE_FT",
    "READY_TO_MOVE",
    "RESALE",
    "LATITUDE",
    "LONGITUDE",
    "LOCALITY_FREQUENCY",
    "AREA_PER_BHK"
]


# ------------------------------------------------------------
# STEP 19: DEFINE CATEGORICAL FEATURES
# ------------------------------------------------------------

categorical_features = [
    "POSTED_BY",
    "BHK_OR_RK",
    "CITY"
]


# ------------------------------------------------------------
# STEP 20: TRAIN-TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


print("\nTraining data shape:")
print(X_train.shape)

print("\nTesting data shape:")
print(X_test.shape)


# ------------------------------------------------------------
# STEP 21: NUMERICAL PREPROCESSING
# ------------------------------------------------------------

numeric_pipeline = Pipeline(
    steps=[

        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )

    ]
)


# ------------------------------------------------------------
# STEP 22: CATEGORICAL PREPROCESSING
# ------------------------------------------------------------

categorical_pipeline = Pipeline(
    steps=[

        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )

    ]
)


# ------------------------------------------------------------
# STEP 23: COMBINE PREPROCESSING
# ------------------------------------------------------------

preprocessor = ColumnTransformer(

    transformers=[

        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),

        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )

    ]
)


# ------------------------------------------------------------
# STEP 24: CREATE MACHINE LEARNING MODEL
# ------------------------------------------------------------

model = RandomForestRegressor(

    n_estimators=200,

    random_state=42,

    n_jobs=-1,

    min_samples_leaf=2
)


# ------------------------------------------------------------
# STEP 25: CREATE COMPLETE PIPELINE
# ------------------------------------------------------------

house_pipeline = Pipeline(

    steps=[

        (
            "preprocessing",
            preprocessor
        ),

        (
            "model",
            model
        )

    ]
)


# ------------------------------------------------------------
# STEP 26: TRAIN MODEL
# ------------------------------------------------------------

print("\nTraining House Price Model...")

house_pipeline.fit(
    X_train,
    y_train
)


print("Model training completed!")


# ------------------------------------------------------------
# STEP 27: MAKE PREDICTIONS
# ------------------------------------------------------------

predictions = house_pipeline.predict(
    X_test
)


# ------------------------------------------------------------
# STEP 28: EVALUATE MODEL
# ------------------------------------------------------------

mae = mean_absolute_error(
    y_test,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        predictions
    )
)

r2 = r2_score(
    y_test,
    predictions
)


print("\n================================")
print("HOUSE PRICE MODEL PERFORMANCE")
print("================================")

print("MAE :", mae)
print("RMSE:", rmse)
print("R2  :", r2)


# ------------------------------------------------------------
# STEP 29: SHOW ACTUAL VS PREDICTED
# ------------------------------------------------------------

comparison = pd.DataFrame({

    "Actual_Price": y_test.values,

    "Predicted_Price": predictions

})


print("\nActual vs Predicted:")
print(
    comparison.head(20)
)


# ------------------------------------------------------------
# STEP 30: SAVE CLEANED DATA
# ------------------------------------------------------------

df.to_csv(
    "house_price_cleaned.csv",
    index=False
)


# ------------------------------------------------------------
# STEP 31: SAVE PREDICTIONS
# ------------------------------------------------------------

comparison.to_csv(
    "house_price_predictions.csv",
    index=False
)


print("\nFiles created:")

print("1. house_price_cleaned.csv")
print("2. house_price_predictions.csv")

print("\nHOUSE PRICE PREPROCESSING COMPLETED!")

Dataset loaded successfully!
Original shape: (29451, 12)

First 5 rows:
  POSTED_BY  UNDER_CONSTRUCTION  RERA  BHK_NO. BHK_OR_RK    SQUARE_FT  \
0     Owner                   0     0        2       BHK  1300.236407   
1    Dealer                   0     0        2       BHK  1275.000000   
2     Owner                   0     0        2       BHK   933.159722   
3     Owner                   0     1        2       BHK   929.921143   
4    Dealer                   1     0        2       BHK   999.009247   

   READY_TO_MOVE  RESALE                      ADDRESS  LONGITUDE   LATITUDE  \
0              1       1        Ksfc Layout,Bangalore  12.969910  77.597960   
1              1       1    Vishweshwara Nagar,Mysore  12.274538  76.644605   
2              1       1             Jigani,Bangalore  12.778033  77.632191   
3              1       1  Sector-1 Vaishali,Ghaziabad  28.642300  77.344500   
4              0       1             New Town,Kolkata  22.592200  88.484911   

   TARGET(PRIC

In [2]:
# ============================================================
# STOCK MARKET TREND ANALYSIS
# COMPLETE DATA PREPROCESSING
# ============================================================


# ------------------------------------------------------------
# STEP 1: IMPORT LIBRARIES
# ------------------------------------------------------------

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


# ------------------------------------------------------------
# STEP 2: LOAD DATASET
# ------------------------------------------------------------

# Google Colab version:
#
# from google.colab import files
# uploaded = files.upload()
# file_path = list(uploaded.keys())[0]

file_path = "stock_market_dataset(1).csv"

df = pd.read_csv(
    file_path
)


print("Dataset loaded successfully!")

print(
    "Original shape:",
    df.shape
)


# ------------------------------------------------------------
# STEP 3: VIEW DATA
# ------------------------------------------------------------

print("\nFirst 5 rows:")
print(
    df.head()
)

print("\nColumn names:")
print(
    df.columns.tolist()
)

print("\nData types:")
print(
    df.dtypes
)

print("\nMissing values:")
print(
    df.isnull().sum()
)

print("\nDuplicate rows:")
print(
    df.duplicated().sum()
)


# ------------------------------------------------------------
# STEP 4: CLEAN COLUMN NAMES
# ------------------------------------------------------------

df.columns = (
    df.columns
    .str.strip()
)


# ------------------------------------------------------------
# STEP 5: CONVERT DATE
# ------------------------------------------------------------

df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce"
)


# Remove invalid dates

df = df.dropna(
    subset=["Date"]
)


# ------------------------------------------------------------
# STEP 6: SORT BY DATE
# ------------------------------------------------------------

df = df.sort_values(
    "Date"
).reset_index(
    drop=True
)


# ------------------------------------------------------------
# STEP 7: REMOVE DUPLICATE DATES
# ------------------------------------------------------------

df = df.drop_duplicates(
    subset=["Date"],
    keep="last"
).reset_index(
    drop=True
)


# ------------------------------------------------------------
# STEP 8: CONVERT PRICE COLUMNS TO NUMERIC
# ------------------------------------------------------------

numeric_columns = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume"
]


for column in numeric_columns:

    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )


# ------------------------------------------------------------
# STEP 9: HANDLE INVALID PRICES
# ------------------------------------------------------------

price_columns = [
    "Open",
    "High",
    "Low",
    "Close"
]


for column in price_columns:

    df.loc[
        df[column] <= 0,
        column
    ] = np.nan


# Volume cannot be negative

df.loc[
    df["Volume"] < 0,
    "Volume"
] = np.nan


# ------------------------------------------------------------
# STEP 10: CHECK OHLC LOGIC
# ------------------------------------------------------------

invalid_ohlc = (

    (df["High"] < df["Open"]) |

    (df["High"] < df["Close"]) |

    (df["Low"] > df["Open"]) |

    (df["Low"] > df["Close"]) |

    (df["High"] < df["Low"])

)


print(
    "\nInvalid OHLC records:",
    invalid_ohlc.sum()
)


# Convert invalid OHLC values to missing

df.loc[
    invalid_ohlc,
    ["Open", "High", "Low", "Close"]
] = np.nan


# ------------------------------------------------------------
# STEP 11: HANDLE MISSING VALUES
# ------------------------------------------------------------

# Financial time series are ordered,
# so forward filling is useful.

df[
    [
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    ]
] = (
    df[
        [
            "Open",
            "High",
            "Low",
            "Close",
            "Volume"
        ]
    ]
    .ffill()
)


# Fill possible missing first observation

df[
    [
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    ]
] = (
    df[
        [
            "Open",
            "High",
            "Low",
            "Close",
            "Volume"
        ]
    ]
    .bfill()
)


# ------------------------------------------------------------
# STEP 12: CREATE DATE FEATURES
# ------------------------------------------------------------

df["Year"] = (
    df["Date"].dt.year
)

df["Month"] = (
    df["Date"].dt.month
)

df["Day"] = (
    df["Date"].dt.day
)

df["DayOfWeek"] = (
    df["Date"].dt.dayofweek
)


# ------------------------------------------------------------
# STEP 13: DAILY RETURN
# ------------------------------------------------------------

df["Daily_Return"] = (
    df["Close"].pct_change()
)


# ------------------------------------------------------------
# STEP 14: LOG RETURN
# ------------------------------------------------------------

df["Log_Return"] = (
    np.log(df["Close"]).diff()
)


# ------------------------------------------------------------
# STEP 15: PRICE RANGE
# ------------------------------------------------------------

df["Price_Range"] = (
    df["High"] -
    df["Low"]
)


# ------------------------------------------------------------
# STEP 16: RANGE PERCENTAGE
# ------------------------------------------------------------

df["Range_Percent"] = (

    (df["High"] - df["Low"])

    / df["Open"]

)


# ------------------------------------------------------------
# STEP 17: OPEN-CLOSE RETURN
# ------------------------------------------------------------

df["Open_Close_Return"] = (

    (df["Close"] - df["Open"])

    / df["Open"]

)


# ------------------------------------------------------------
# STEP 18: LAG FEATURES
# ------------------------------------------------------------

df["Close_Lag_1"] = (
    df["Close"].shift(1)
)

df["Close_Lag_2"] = (
    df["Close"].shift(2)
)

df["Close_Lag_3"] = (
    df["Close"].shift(3)
)


df["Return_Lag_1"] = (
    df["Daily_Return"].shift(1)
)


df["Volume_Lag_1"] = (
    df["Volume"].shift(1)
)


# ------------------------------------------------------------
# STEP 19: MOVING AVERAGES
# ------------------------------------------------------------

# shift(1) prevents data leakage.

df["MA_3"] = (

    df["Close"]
    .shift(1)
    .rolling(3)
    .mean()

)


df["MA_5"] = (

    df["Close"]
    .shift(1)
    .rolling(5)
    .mean()

)


df["MA_10"] = (

    df["Close"]
    .shift(1)
    .rolling(10)
    .mean()

)


# ------------------------------------------------------------
# STEP 20: VOLATILITY
# ------------------------------------------------------------

df["Volatility_5"] = (

    df["Daily_Return"]
    .shift(1)
    .rolling(5)
    .std()

)


df["Volatility_10"] = (

    df["Daily_Return"]
    .shift(1)
    .rolling(10)
    .std()

)


# ------------------------------------------------------------
# STEP 21: VOLUME MOVING AVERAGE
# ------------------------------------------------------------

df["Volume_MA_5"] = (

    df["Volume"]
    .shift(1)
    .rolling(5)
    .mean()

)


# ------------------------------------------------------------
# STEP 22: CREATE FUTURE RETURN
# ------------------------------------------------------------

# Tomorrow's return

df["Next_Day_Return"] = (

    df["Close"].shift(-1)
    / df["Close"]

) - 1


# ------------------------------------------------------------
# STEP 23: CREATE TARGET
# ------------------------------------------------------------

# 1 = price goes UP tomorrow
# 0 = price goes DOWN tomorrow

df["Target_Direction"] = (

    df["Next_Day_Return"] > 0

).astype(int)


# ------------------------------------------------------------
# STEP 24: REMOVE LAST ROW
# ------------------------------------------------------------

# Last row does not have tomorrow's price.

df = df.iloc[:-1].copy()


# ------------------------------------------------------------
# STEP 25: REMOVE ROWS WITHOUT ENOUGH HISTORY
# ------------------------------------------------------------

df = df.dropna(
    subset=[
        "Close_Lag_1",
        "Close_Lag_2",
        "Close_Lag_3",
        "MA_3",
        "MA_5",
        "MA_10",
        "Volatility_5",
        "Volatility_10"
    ]
)


df = df.reset_index(
    drop=True
)


# ------------------------------------------------------------
# STEP 26: DISPLAY PROCESSED DATA
# ------------------------------------------------------------

print("\nProcessed dataset:")

print(
    df.head()
)

print(
    "\nProcessed shape:",
    df.shape
)


# ------------------------------------------------------------
# STEP 27: DEFINE FEATURES
# ------------------------------------------------------------

features = [

    "Open",
    "High",
    "Low",
    "Close",
    "Volume",

    "Year",
    "Month",
    "Day",
    "DayOfWeek",

    "Daily_Return",
    "Log_Return",

    "Price_Range",
    "Range_Percent",
    "Open_Close_Return",

    "Close_Lag_1",
    "Close_Lag_2",
    "Close_Lag_3",

    "Return_Lag_1",
    "Volume_Lag_1",

    "MA_3",
    "MA_5",
    "MA_10",

    "Volatility_5",
    "Volatility_10",

    "Volume_MA_5"

]


target = "Target_Direction"


# ------------------------------------------------------------
# STEP 28: CHRONOLOGICAL TRAIN-TEST SPLIT
# ------------------------------------------------------------

# DO NOT randomly shuffle stock data.

split_index = int(
    len(df) * 0.80
)


train = df.iloc[
    :split_index
].copy()


test = df.iloc[
    split_index:
].copy()


X_train = train[
    features
]

y_train = train[
    target
]


X_test = test[
    features
]

y_test = test[
    target
]


print("\nTraining data:")
print(
    X_train.shape
)

print("\nTesting data:")
print(
    X_test.shape
)


print("\nTraining date range:")

print(
    train["Date"].min(),
    "to",
    train["Date"].max()
)


print("\nTesting date range:")

print(
    test["Date"].min(),
    "to",
    test["Date"].max()
)


# ------------------------------------------------------------
# STEP 29: CREATE PREPROCESSING PIPELINE
# ------------------------------------------------------------

preprocessor = Pipeline(

    steps=[

        (
            "imputer",

            SimpleImputer(
                strategy="median"
            )
        )

    ]

)


# ------------------------------------------------------------
# STEP 30: CREATE MODEL
# ------------------------------------------------------------

model = RandomForestClassifier(

    n_estimators=200,

    max_depth=5,

    min_samples_leaf=2,

    random_state=42,

    class_weight="balanced"

)


# ------------------------------------------------------------
# STEP 31: COMPLETE PIPELINE
# ------------------------------------------------------------

stock_pipeline = Pipeline(

    steps=[

        (
            "preprocessing",
            preprocessor
        ),

        (
            "model",
            model
        )

    ]

)


# ------------------------------------------------------------
# STEP 32: TRAIN MODEL
# ------------------------------------------------------------

print(
    "\nTraining Stock Market Model..."
)


stock_pipeline.fit(
    X_train,
    y_train
)


print(
    "Training completed!"
)


# ------------------------------------------------------------
# STEP 33: MAKE PREDICTIONS
# ------------------------------------------------------------

predictions = stock_pipeline.predict(
    X_test
)


# Probability that price will increase

probabilities = (

    stock_pipeline
    .predict_proba(X_test)[:, 1]

)


# ------------------------------------------------------------
# STEP 34: MODEL EVALUATION
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_test,
    predictions
)


print("\n================================")
print("STOCK MARKET MODEL PERFORMANCE")
print("================================")

print(
    "Accuracy:",
    accuracy
)


print("\nClassification Report:")

print(
    classification_report(
        y_test,
        predictions,
        zero_division=0
    )
)


print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        predictions
    )
)


# ------------------------------------------------------------
# STEP 35: CREATE RESULT TABLE
# ------------------------------------------------------------

results = test[
    [
        "Date",
        "Close"
    ]
].copy()


results["Actual_Direction"] = (
    y_test.values
)


results["Predicted_Direction"] = (
    predictions
)


results["Probability_Up"] = (
    probabilities
)


results["Actual_Next_Day_Return"] = (
    test["Next_Day_Return"].values
)


print("\nPrediction results:")

print(
    results
)


# ------------------------------------------------------------
# STEP 36: SAVE PROCESSED DATA
# ------------------------------------------------------------

df.to_csv(
    "stock_market_processed.csv",
    index=False
)


# ------------------------------------------------------------
# STEP 37: SAVE PREDICTIONS
# ------------------------------------------------------------

results.to_csv(
    "stock_market_predictions.csv",
    index=False
)


print("\nFiles created:")

print(
    "1. stock_market_processed.csv"
)

print(
    "2. stock_market_predictions.csv"
)


print(
    "\nSTOCK MARKET PREPROCESSING COMPLETED!"
)

FileNotFoundError: [Errno 2] No such file or directory: 'stock_market_dataset(1).csv'